# Antimicrobial Peptide Activity Prediction (E. coli) - Fully Commented

## Imports

In [ ]:
# "import" loads a ready-made toolbox (a library) so we can use its functions.
# Python alone cannot read tables or train models, so we borrow these tools.

# re = "regular expressions". A tool for checking text against a pattern.
# We use it to check that a sequence contains only valid amino-acid letters.
import re

# numpy = fast maths on numbers. "as np" gives it the short nickname np.
# We use it for np.nan (the value meaning "missing") and for random shuffling.
import numpy as np

# pandas = tables of data (like Excel inside Python). Nickname pd.
# A pandas table is called a DataFrame. We use it to hold the peptide data.
import pandas as pd

# matplotlib.pyplot = draws graphs. Nickname plt.
# We use it to show the confusion matrices and the bar chart.
import matplotlib.pyplot as plt

# seaborn = makes matplotlib graphs look nicer. Nickname sns.
import seaborn as sns

# peptides = a biology toolbox. Give it a sequence and it calculates
# properties such as charge, hydrophobicity and molecular weight.
import peptides

# "from X import Y" means: take only the piece Y out of toolbox X.
# scikit-learn (sklearn) is the machine-learning toolbox. Pieces we take:

# train_test_split = cuts the data into a training part and a test part.
from sklearn.model_selection import train_test_split

# SVC = Support Vector Classifier, the SVM model (draws a boundary between classes).
from sklearn.svm import SVC

# RandomForestClassifier = many decision trees that vote on the answer.
from sklearn.ensemble import RandomForestClassifier

# GaussianNB = Gaussian Naive Bayes, a model that works with probabilities.
from sklearn.naive_bayes import GaussianNB

# make_pipeline = joins several steps into one model (e.g. scale, then classify).
from sklearn.pipeline import make_pipeline

# StandardScaler = rescales every feature to a similar size.
# Needed because SVM is thrown off when one feature has much bigger numbers than another.
from sklearn.preprocessing import StandardScaler

# clone = makes a fresh, untrained copy of a model with the same settings.
from sklearn.base import clone

# The scoring tools: how many predictions were right, how many alarms were
# false, how many real actives were found, and a balance of the last two (F1).
# ConfusionMatrixDisplay draws the 2x2 box of correct/wrong predictions.
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)

# set_theme sets the look of all later charts once.
# style="whitegrid" = white background with faint grid lines.
sns.set_theme(style="whitegrid")

## Configuration

In [ ]:
# These are SETTINGS. Written once here so you never hunt through the code.

# The name of your data file. It must be in the same folder as this notebook.
CSV_PATH = "dbaasp_ecoli.csv"

# The name of the column that holds the amino-acid sequence.
# If your file calls it something else, change it here.
SEQ_COL  = "SEQUENCE"   # change to match your file

# The name of the column that holds the MIC value.
# In this dataset the values are already in ug/mL.
MIC_COL  = "MIC_µgml"  # already in ug/mL in this dataset

# The name of the column that holds the unit (ug/mL or uM).
# This dataset has no such column, so the code assumes ug/mL.
UNIT_COL = "UNIT"        # not present here -> values assumed ug/mL

# The cutoff between active and inactive.
# MIC is the smallest dose that stops bacteria growing, so SMALLER = stronger.
# MIC of 32 or less counts as active. 32 is a chosen convention, not a law.
MIC_THRESHOLD = 32           # ug/mL; active if MIC <= threshold

# The 20 standard amino-acid letters. Used to check sequences are valid
# and to count how much of each amino acid a peptide contains.
AA = "ACDEFGHIKLMNPQRSTVWY"

# A fixed number that "freezes" every random step (the split, the forest,
# the shuffle). With it, you get the SAME results every time you run the notebook.
# 42 is a popular choice; any number works.
RANDOM_STATE = 42

## Load data

In [ ]:
# pd.read_csv reads a CSV file (a text file of comma-separated values)
# and turns it into a pandas table (DataFrame). We name the table df.
df = pd.read_csv(CSV_PATH)

# df.columns = the column headings of the table.
# list(...) turns them into a plain list. print shows it on screen.
# Check this against the names in the Configuration cell.
print("Columns:", list(df.columns))

# df.shape = (number of rows, number of columns).
# Rows are peptides, so the first number is how many peptides you loaded.
print("Shape:", df.shape)

# df.head() shows the first 5 rows so you can see what the data looks like.
# It is the last line of the cell, so Jupyter displays it automatically.
df.head()

In [ ]:
# This cell checks your column names are correct. If not, it stops with a
# clear message instead of failing later in a confusing way.

# A dictionary (curly brackets) pairs a label with the column name you chose.
# "sequence" -> SEQ_COL, "MIC" -> MIC_COL.
required = {"sequence": SEQ_COL, "MIC": MIC_COL}

# A list comprehension: a short way to build a list with a loop.
# "for role, col in required.items()" goes through each pair.
# "if col not in df.columns" keeps only the names that do NOT exist in the file.
# The f"..." is an f-string: values inside { } are filled into the text.
missing = [f"{role} -> '{col}'" for role, col in required.items() if col not in df.columns]

# An empty list counts as False; a list with items counts as True.
# So the block below runs only when something is missing.
if missing:
    # raise KeyError(...) stops the notebook and shows a red error message.
    raise KeyError(
        # "; ".join(missing) glues the missing names together with semicolons.
        "Column(s) not found: " + "; ".join(missing)
        # Shows the columns that DO exist, so you can pick the right one.
        # "\n" starts a new line inside the message.
        + f"\nAvailable columns: {list(df.columns)}"
        # Tells you exactly how to fix it.
        + "\nEdit SEQ_COL / MIC_COL in the Configuration cell to match."
    )

# "in" checks membership. This gives True if the file has a unit column, else False.
# We store it because a later step behaves differently with or without units.
has_units = UNIT_COL in df.columns

# Show the answer so you know which case you are in.
print("Unit column present:", has_units)

## Clean & filter

In [ ]:
# Real databases are messy. Before analysing, keep only rows we can trust.

# re.compile builds a reusable text pattern. Reading the pattern:
#   ^       = the start of the text
#   [ ... ] = any ONE of the letters inside (here the 20 amino acids from AA)
#   +       = one or more of those letters in a row
#   $       = the end of the text
# So it matches only sequences made entirely of the 20 standard letters.
valid_seq = re.compile(f"^[{AA}]+$")

# "def" defines your own function. This one is called first_number and takes x.
# Purpose: pull the first number out of a value, e.g. ">32" becomes 32.
def first_number(x):
    # str(x) makes sure x is text. re.search looks for the first piece of text
    # matching the pattern: optional sign, optional digits, optional decimal
    # point, then digits. r"..." is a raw string so backslashes are kept as typed.
    m = re.search(r"[-+]?\d*\.?\d+", str(x))
    # If a match was found (m is not empty): return it as a decimal number.
    # float() converts text to a decimal number; m.group() is the matched text.
    # Otherwise return np.nan, which means "missing".
    return float(m.group()) if m else np.nan

# .copy() makes a separate copy, so our edits never change the original df.
data = df.copy()

# Tidy the sequence text in three steps chained with dots:
#   .astype(str)  = make every value text
#   .str.strip()  = remove spaces at the start and end
#   .str.upper()  = make all letters capital, so "k" and "K" are the same amino acid
data[SEQ_COL] = data[SEQ_COL].astype(str).str.strip().str.upper()

# Keep only rows with a valid sequence.
#   .apply(...) runs a function on every value in the column.
#   lambda s: ... is a tiny one-line function: "for sequence s, do this".
#   valid_seq.match(s) checks s against the pattern; bool() turns it into True/False.
#   data[ True/False list ] keeps only the rows marked True.
data = data[data[SEQ_COL].apply(lambda s: bool(valid_seq.match(s)))]

# Make a NEW column holding the MIC as a clean number, using first_number.
data["MIC_value"] = data[MIC_COL].apply(first_number)

# Keep rows where the MIC exists AND is above zero.
#   .notna() = "is not missing".
#   &        = "and" (both conditions must hold).
#   A concentration cannot be zero or negative, so those rows are bad data.
data = data[data["MIC_value"].notna() & (data["MIC_value"] > 0)]

# len(data) = number of rows left. Shows how many peptides survived cleaning.
print("Rows with valid sequence and numeric MIC:", len(data))

## Standardise MIC units to µg/mL

In [ ]:
# MIC can be reported in two units that CANNOT be compared directly:
#   ug/mL = a MASS (micrograms of peptide per millilitre)
#   uM    = a COUNT of molecules (micromolar)
# To convert uM to ug/mL we need the molecular weight (how heavy one molecule is).

# A function that turns a messy unit label into a standard one.
def norm_unit(u):
    # Chain of tidy-ups on the label:
    #   str(u)          = make it text
    #   .strip()        = remove surrounding spaces
    #   .lower()        = lowercase everything
    #   .replace(a, b)  = swap a for b. Here both micro symbols (µ and
    #                     μ) become a plain "u", so uM, µM and μM all match.
    u = str(u).strip().lower().replace("µ", "u").replace("μ", "u")
    # { ... } is a set of accepted spellings. "in" checks if u is one of them.
    # All of these mean micrograms per millilitre.
    if u in {"ug/ml", "mg/l", "ug/cm3"}:
        # Return the standard label.
        return "ug/mL"
    # All of these mean micromolar.
    if u in {"um", "umol/l", "umol/ml"}:
        return "uM"
    # None means "nothing / unrecognised". Those rows get dropped later.
    return None

# Make a tidy unit column.
# "A if condition else B" picks A when the condition is True, otherwise B.
# If the file has a unit column, tidy each value with norm_unit.
# If not, assume everything is already ug/mL.
data["unit_std"] = data[UNIT_COL].apply(norm_unit) if has_units else "ug/mL"

# Molecular weight of every peptide.
# peptides.Peptide(s) wraps sequence s in an object that knows its chemistry.
# .molecular_weight() asks that object for its weight (in g/mol).
data["MW"] = data[SEQ_COL].apply(lambda s: peptides.Peptide(s).molecular_weight())

# A function that converts ONE row's MIC to ug/mL. "row" is one peptide's data.
def to_ugml(row):
    # Already ug/mL: keep the value unchanged.
    if row["unit_std"] == "ug/mL":
        return row["MIC_value"]
    # Micromolar: convert with  ug/mL = uM x molecular weight / 1000.
    # Units: uM x (g/mol) gives a mass; dividing by 1000 fixes the scale.
    if row["unit_std"] == "uM":
        return row["MIC_value"] * row["MW"] / 1000.0   # uM * g/mol / 1000 = ug/mL
    # Unit missing or unknown: we cannot convert, so mark as missing.
    return np.nan

# Apply to_ugml to every row.
# axis=1 means "go row by row" (axis=0 would go column by column).
data["MIC_ugml"] = data.apply(to_ugml, axis=1)

# .isna() marks missing values as True; .sum() counts the Trues;
# int(...) makes it a whole number. This counts rows that could not be converted.
dropped = int(data["MIC_ugml"].isna().sum())

# Keep only rows that DO have a converted value (drop the rest).
data = data[data["MIC_ugml"].notna()]

# Report what happened. In an f-string, {dropped} is replaced by its value.
print(f"Dropped {dropped} row(s) with missing/unconvertible units.")
print("Remaining rows:", len(data))

## Label & class balance

In [ ]:
# The models need a yes/no ANSWER to learn. That answer is called the label.

# (data["MIC_ugml"] <= MIC_THRESHOLD) gives True/False for every peptide:
# True when the MIC is 32 or less (strong enough = active).
# .astype(int) turns True into 1 (active) and False into 0 (inactive).
data["label"] = (data["MIC_ugml"] <= MIC_THRESHOLD).astype(int)

# .value_counts() counts how many peptides have each label.
# .sort_index() puts label 0 before label 1 so it reads in order.
counts = data["label"].value_counts().sort_index()

# Show the counts with a heading.
print("Class balance (0 = inactive, 1 = active):")
print(counts)

# Ratio of the bigger class to the smaller class.
# 1.0 = perfectly even; 5.0 = one class is five times the other.
ratio = counts.max() / counts.min()

# Our rule: if one class is at least twice the other, call it imbalanced.
# Result is True/False and is used later to switch on class_weight.
# Why it matters: with 90% inactive, always answering "inactive" scores
# 90% accuracy while finding zero active peptides.
imbalanced = ratio >= 2

# {ratio:.2f} shows the ratio with 2 decimals.
# "'imbalanced' if imbalanced else 'balanced'" picks the matching word.
print(f"Imbalance ratio: {ratio:.2f} -> {'imbalanced' if imbalanced else 'balanced'}")

## Features

In [ ]:
# Models only understand numbers, not letters. A FEATURE is one number that
# describes a peptide. This function turns ONE sequence into a set of numbers.

def features(seq):
    # Wrap the sequence so we can ask it for chemical properties.
    p = peptides.Peptide(seq)

    # A dictionary: feature name -> its value.
    f = {
        # len() counts the amino acids. Length affects how a peptide spans a membrane.
        "length": len(seq),
        # Net electric charge. AMPs are usually positive, so they are drawn to the
        # negatively charged bacterial membrane.
        "net_charge": p.charge(),
        # Average "water-hating" score. Oily parts let the peptide push into the
        # oily core of the membrane.
        "hydrophobicity": p.hydrophobicity(),
        # How strongly oily and charged residues sit on OPPOSITE faces of the
        # peptide (amphipathic). This depends on ORDER, not just composition.
        # "or 0.0" = if nothing is returned (very short peptides), use 0 instead.
        "hydrophobic_moment": p.hydrophobic_moment() or 0.0,
        # The pH at which the peptide has no net charge.
        "isoelectric_point": p.isoelectric_point(),
        # Share of small oily residues (A, V, I, L); linked to stability.
        "aliphatic_index": p.aliphatic_index(),
    }

    # Loop over each of the 20 amino-acid letters (a is one letter each time).
    for a in AA:
        # seq.count(a) = how many times letter a appears.
        # Divide by length, multiply by 100 = percentage composition.
        # f"pct_{a}" builds the feature name, e.g. "pct_K".
        f[f"pct_{a}"] = 100.0 * seq.count(a) / len(seq)

    # Give the finished dictionary back to whoever called the function.
    return f

# Build the feature table X (capital X = the INPUTS, by convention).
# [features(s) for s in data[SEQ_COL]] runs features() on every sequence.
# pd.DataFrame(...) stacks the results into a table: one row per peptide.
# index=data.index keeps the row labels aligned with the original data.
X = pd.DataFrame([features(s) for s in data[SEQ_COL]], index=data.index)

# y (lowercase y = the ANSWERS, by convention) is the 0/1 label column.
y = data["label"]

# Show the table size: (number of peptides, number of features).
print("Feature matrix:", X.shape)

# Preview the first 5 rows of the feature table.
X.head()

## Train / test split

In [ ]:
# Hide some peptides from the model, then test on them. This checks real
# learning. Testing on peptides it already saw would let it just memorise
# the answers (overfitting) and give a falsely high score.

# train_test_split returns FOUR pieces, always in this order:
#   X_train = inputs for learning     X_test = inputs for the exam
#   y_train = answers for learning    y_test = answers for the exam
X_train, X_test, y_train, y_test = train_test_split(
    # The full inputs and answers to be cut up.
    # test_size=0.20  -> 20% go to the test set, 80% to training.
    # stratify=y      -> keep the same active/inactive proportion in BOTH parts,
    #                    so luck cannot put most active peptides on one side.
    # random_state    -> fixed seed, so the same split happens on every run.
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# .shape shows (rows, columns) of each part. Train should be about 4x bigger.
print("Train:", X_train.shape, " Test:", X_test.shape)

## Models

In [ ]:
# class_weight="balanced" tells a model to pay MORE attention to the rarer class.
# We only switch it on if the classes were imbalanced (decided earlier).
# None means "no adjustment".
cw = "balanced" if imbalanced else None

# A dictionary that holds the three models by name.
# They are only CREATED here; they have not learned anything yet.
models = {
    # make_pipeline chains two steps into one model:
    #   1) StandardScaler() rescales all features to a similar range, because
    #      an SVM measures distances and big-number features would dominate.
    #   2) SVC(...) is the SVM. It draws the widest possible boundary between
    #      active and inactive peptides.
    # class_weight = see above. random_state = fixed seed.
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight=cw, random_state=RANDOM_STATE)),
    # Random Forest = a forest of many decision trees that VOTE; the majority wins.
    # It needs no scaling because trees only ask "is this number above X?".
    "Random Forest": RandomForestClassifier(class_weight=cw, random_state=RANDOM_STATE),
    # Gaussian Naive Bayes = learns what each feature typically looks like for
    # active vs inactive peptides and combines the clues as probabilities.
    # It has no class_weight option and needs no scaling.
    "Gaussian NB": GaussianNB(),
}

In [ ]:
# An empty dictionary to collect each model's scores for the comparison table.
results = {}

# Loop over the three models. models.items() gives (name, model) pairs.
for name, model in models.items():
    # fit = TRAIN. The model studies X_train together with the true answers
    # y_train and adjusts itself to find the pattern.
    model.fit(X_train, y_train)

    # predict = GUESS. Give the model the unseen test peptides (no answers)
    # and it returns a 0/1 guess for each one.
    pred = model.predict(X_test)

    # Compare the guesses (pred) with the true answers (y_test):
    results[name] = {
        # Accuracy  = share of ALL guesses that were right.
        "Accuracy": accuracy_score(y_test, pred),
        # Precision = of peptides called active, the share that truly are.
        # (High = few false alarms.) zero_division=0: if the maths would divide
        # by zero, give 0 instead of a warning.
        "Precision": precision_score(y_test, pred, zero_division=0),
        # Recall = of peptides that truly are active, the share the model found.
        # (High = few real antibiotics missed.)
        "Recall": recall_score(y_test, pred, zero_division=0),
        # F1 = one number balancing precision and recall.
        "F1": f1_score(y_test, pred, zero_division=0),
    }

    # f"\n=== {name} ===" prints a blank line (\n) then the model's name.
    print(f"\n=== {name} ===")

    # Go through this model's four scores one by one.
    for metric, value in results[name].items():
        # {metric:10s} = name padded to 10 characters so numbers line up.
        # {value:.3f}  = number shown with 3 decimals.
        print(f"{metric:10s}: {value:.3f}")

    # Draw the confusion matrix: a 2x2 box counting right and wrong guesses.
    # display_labels gives names to the 0 and 1 rows/columns.
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["inactive", "active"])

    # Put the model's name as the chart title.
    plt.title(name)

    # Display the chart now (otherwise charts can pile up).
    plt.show()

## Model comparison

In [ ]:
# pd.DataFrame(results) makes a table with models as COLUMNS.
# .T ("transpose") flips it so each ROW is a model and each column a score.
# [["Accuracy", ...]] picks the columns and sets their left-to-right order.
scores = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1"]]

# .round(3) shows 3 decimals. Last line of the cell, so it is displayed.
scores.round(3)

## Random Forest feature importance

In [ ]:
# Pull the already-trained Random Forest out of the dictionary.
rf = models["Random Forest"]

# rf.feature_importances_ = how much each feature helped the forest decide
# (bigger = more useful). Steps chained with dots:
#   pd.Series(values, index=names) = attach feature names to the numbers
#   .sort_values(ascending=False)  = biggest first
#   .head(10)                      = keep the top 10
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

# Start a new chart, 7 inches wide and 4 inches tall.
plt.figure(figsize=(7, 4))

# Horizontal bar chart. x = bar length (importance), y = bar label (feature name).
sns.barplot(x=imp.values, y=imp.index, color="steelblue")

# Label the horizontal axis.
plt.xlabel("Importance")

# Give the chart a title.
plt.title("Top 10 features (Random Forest)")

# Adjust spacing so labels are not cut off.
plt.tight_layout()

# Display the chart.
plt.show()

## Does arrangement matter? Shuffled-sequence control

In [ ]:
# Question: does the ORDER of amino acids matter, or only WHICH ones are present?
# Test: shuffle each sequence (same letters, random order) and retrain.
# Big score drop = order matters. Small drop = mostly composition mattered.

# .idxmax() returns the row name (model name) holding the highest F1.
best_name = scores["F1"].idxmax()
print("Best model by F1:", best_name)

# A random-number generator with a fixed seed, so shuffles repeat identically.
rng = np.random.default_rng(RANDOM_STATE)

# Function that scrambles one sequence.
def shuffle_seq(seq):
    # list("ABC") = ["A","B","C"]: split the sequence into separate letters.
    chars = list(seq)
    # Shuffle the letters into random order, in place.
    rng.shuffle(chars)
    # "".join(...) glues the letters back into one string.
    return "".join(chars)

# Shuffle every peptide. Composition is identical; only the order changes.
shuffled = data[SEQ_COL].apply(shuffle_seq)

# Rebuild all features from the shuffled sequences with the SAME function.
# Length, charge and composition stay the same; hydrophobic moment can change.
X_shuf = pd.DataFrame([features(s) for s in shuffled], index=data.index)

# Split the shuffled data exactly as before (same settings), so the
# comparison with the original result is fair.
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_shuf, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

# clone = a brand-new, untrained copy of the winning model.
# We must start fresh, otherwise it would still remember the original data.
m = clone(models[best_name])

# Train the fresh copy on the shuffled data.
m.fit(Xs_train, ys_train)

# F1 score of this model on the shuffled test set.
# m.predict(Xs_test) = its guesses; ys_test = the true answers.
shuf_f1 = f1_score(ys_test, m.predict(Xs_test), zero_division=0)

# The same model's F1 on the real, unshuffled data.
# .loc[row, column] picks one cell from the scores table.
orig_f1 = scores.loc[best_name, "F1"]

# Report both and the difference. A larger drop means order mattered more.
print(f"Original F1 : {orig_f1:.3f}")
print(f"Shuffled F1 : {shuf_f1:.3f}")
print(f"Drop        : {orig_f1 - shuf_f1:.3f}")